# Lab 6 - Advanced Preprocessing & CNNs
## Version 1

This notebook completes the required lab tasks:
- perspective correction
- automatic deskewing
- morphological operations
- CNN training on MNIST
- training history plot
- test prediction visualization

## Setup

```bash
pip install tensorflow keras opencv-python scikit-image imutils matplotlib
```

In [ ]:
import os
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import mnist

np.random.seed(42)

DATA_DIR = Path("week6_demo_images")
DATA_DIR.mkdir(exist_ok=True)

def show_image(title, image, cmap=None, figsize=(6, 5)):
    plt.figure(figsize=figsize)
    if len(image.shape) == 2:
        plt.imshow(image, cmap=cmap or "gray")
    else:
        plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

def create_demo_document(path=DATA_DIR / "document_flat.png"):
    img = np.full((700, 520, 3), 255, dtype=np.uint8)
    cv2.rectangle(img, (35, 35), (485, 665), (245, 245, 245), -1)
    cv2.rectangle(img, (35, 35), (485, 665), (40, 40, 40), 2)
    cv2.putText(img, "LAB 6 OCR DOCUMENT", (70, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (20, 20, 20), 2)
    lines = [
        "Receipt No: 2048",
        "Date: 2026-04-26",
        "Perspective correction",
        "Automatic deskewing",
        "Morphological operations",
        "CNN digit recognition"
    ]
    y = 150
    for line in lines:
        cv2.putText(img, line, (70, y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (30, 30, 30), 1, cv2.LINE_AA)
        y += 60
    cv2.imwrite(str(path), img)
    return img

def create_demo_images():
    flat = create_demo_document()
    h, w = flat.shape[:2]

    src = np.float32([[35, 35], [485, 35], [485, 665], [35, 665]])
    dst = np.float32([[90, 10], [450, 80], [500, 650], [20, 610]])
    matrix = cv2.getPerspectiveTransform(src, dst)
    perspective = cv2.warpPerspective(flat, matrix, (w, h), borderValue=(235, 235, 235))
    cv2.imwrite(str(DATA_DIR / "document_perspective.png"), perspective)

    rotation = cv2.getRotationMatrix2D((w // 2, h // 2), 7, 1.0)
    skewed = cv2.warpAffine(flat, rotation, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    cv2.imwrite(str(DATA_DIR / "skewed_document.jpg"), skewed)

    gray = cv2.cvtColor(flat, cv2.COLOR_BGR2GRAY)
    noise = np.random.normal(0, 22, gray.shape).astype(np.int16)
    noisy = np.clip(gray.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    cv2.imwrite(str(DATA_DIR / "noisy_document.jpg"), noisy)

create_demo_images()
print("Images ready:", DATA_DIR.resolve())

## Task 1.1 - Perspective Correction

In [ ]:
def order_points(pts):
    rect = np.zeros((4, 2), dtype="float32")
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]
    return rect

def four_point_transform(image, pts):
    rect = order_points(pts)
    tl, tr, br, bl = rect

    width_a = np.linalg.norm(br - bl)
    width_b = np.linalg.norm(tr - tl)
    max_width = max(int(width_a), int(width_b))

    height_a = np.linalg.norm(tr - br)
    height_b = np.linalg.norm(tl - bl)
    max_height = max(int(height_a), int(height_b))

    dst = np.array([
        [0, 0],
        [max_width - 1, 0],
        [max_width - 1, max_height - 1],
        [0, max_height - 1]
    ], dtype="float32")

    matrix = cv2.getPerspectiveTransform(rect, dst)
    warped = cv2.warpPerspective(image, matrix, (max_width, max_height))
    return warped

image = cv2.imread(str(DATA_DIR / "document_perspective.png"))
points = np.array([[90, 10], [450, 80], [500, 650], [20, 610]], dtype="float32")
corrected = four_point_transform(image, points)

show_image("Original", image)
show_image("Perspective corrected", corrected)

## Task 1.2 - Automatic Deskewing

In [ ]:
def deskew(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray = cv2.bitwise_not(gray)
    thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)[1]

    coords = np.column_stack(np.where(thresh > 0))
    angle = cv2.minAreaRect(coords)[-1]

    if angle < -45:
        angle = -(90 + angle)
    else:
        angle = -angle

    h, w = image.shape[:2]
    center = (w // 2, h // 2)
    matrix = cv2.getRotationMatrix2D(center, angle, 1.0)

    rotated = cv2.warpAffine(
        image,
        matrix,
        (w, h),
        flags=cv2.INTER_CUBIC,
        borderMode=cv2.BORDER_REPLICATE
    )
    return rotated, angle

skewed = cv2.imread(str(DATA_DIR / "skewed_document.jpg"))
deskewed, angle = deskew(skewed)

print(f"Detected angle: {angle:.2f}")
show_image("Skewed", skewed)
show_image("Deskewed", deskewed)

## Task 1.3 - Morphological Operations

In [ ]:
img = cv2.imread(str(DATA_DIR / "noisy_document.jpg"))
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img

kernel = np.ones((3, 3), np.uint8)

erosion = cv2.erode(gray, kernel, iterations=1)
dilation = cv2.dilate(gray, kernel, iterations=1)
opening = cv2.morphologyEx(gray, cv2.MORPH_OPEN, kernel)
closing = cv2.morphologyEx(gray, cv2.MORPH_CLOSE, kernel)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
images = [
    ("Original", gray),
    ("Erosion", erosion),
    ("Dilation", dilation),
    ("Opening", opening),
    ("Closing", closing)
]

for ax, (title, item) in zip(axes.ravel(), images):
    ax.imshow(item, cmap="gray")
    ax.set_title(title)
    ax.axis("off")

axes.ravel()[-1].axis("off")
plt.tight_layout()
plt.show()

## Task 2.1 - Load MNIST Dataset

In [ ]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

print(f"Training samples: {x_train.shape[0]}")
print(f"Test samples: {x_test.shape[0]}")
print(f"Image shape: {x_train.shape[1:]}")

plt.figure(figsize=(10, 3))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i], cmap="gray")
    plt.title(f"Label: {y_train[i]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

## Task 2.2 - Preprocess Data

In [ ]:
x_train = x_train.reshape(-1, 28, 28, 1).astype("float32") / 255
x_test = x_test.reshape(-1, 28, 28, 1).astype("float32") / 255

y_train_cat = keras.utils.to_categorical(y_train, 10)
y_test_cat = keras.utils.to_categorical(y_test, 10)

print(x_train.shape)
print(y_train_cat.shape)

## Task 2.3 - Build CNN Architecture

In [ ]:
model = keras.Sequential([
    layers.Conv2D(32, (3, 3), activation="relu", input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(10, activation="softmax")
])

model.summary()

## Task 2.4 - Train the Model

In [ ]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    x_train,
    y_train_cat,
    batch_size=128,
    epochs=10,
    validation_split=0.1
)

test_loss, test_acc = model.evaluate(x_test, y_test_cat, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")

## Training History

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.title("Training History")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

## Test Predictions

In [ ]:
predictions = model.predict(x_test[:25])
predicted_labels = np.argmax(predictions, axis=1)

plt.figure(figsize=(12, 12))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.imshow(x_test[i].reshape(28, 28), cmap="gray")
    plt.title(f"Pred: {predicted_labels[i]}\nTrue: {y_test[i]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

## Analysis

Perspective correction straightens documents photographed from an angle. Deskewing fixes rotated scans. Morphological operations help reduce noise and improve document quality. The CNN learns digit features through convolution and pooling layers, then classifies each digit with dense layers.